In [4]:
from pathlib import Path

import pandas as pd
import pypsa

INPUT_DIR = Path("inputs") / "tamil_nadu_ra_2025_26"
n_year = pypsa.Network(INPUT_DIR)
component_metadata = pd.read_csv(INPUT_DIR / "component_metadata.csv")
plant_generators = component_metadata.loc[
    component_metadata.component_type == "Generator", "component_id"
]
plant_storage = component_metadata.loc[
    component_metadata.component_type == "StorageUnit", "component_id"
]
assert set(plant_generators) <= set(n_year.generators.index)
assert set(plant_storage) == set(n_year.storage_units.index)
print(f"Loaded {len(plant_generators)} workbook-record generators and {len(plant_storage)} storage units")


# Select the Monday-Sunday week containing the annual demand peak.
annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[(n_year.snapshots >= week_start) & (n_year.snapshots < week_end)]

assert len(week) == 168, f"Expected 168 hourly snapshots, found {len(week)}"
n = n_year.copy(snapshots=week)

print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(f"Optimizing peak week: {week_start} to {week_end - pd.Timedelta(hours=1)}")

INFO:pypsa.network.io:Imported network 'Tamil Nadu FY 2025-26 single-node UC inputs' has buses, carriers, generators, loads, storage_units


Loaded 276 workbook-record generators and 4 storage units
Annual peak: 19,987.33 MW at 2025-07-11 16:00:00
Optimizing peak week: 2025-07-07 00:00:00 to 2025-07-13 23:00:00


In [5]:
n

PyPSA Network 'Tamil Nadu FY 2025-26 single-node UC inputs'
-----------------------------------------------------------
Components:
 - Bus: 1
 - Carrier: 12
 - Generator: 278
 - Load: 1
 - StorageUnit: 4
Snapshots: 168

Model overview

In [6]:
display(n.generators)

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
coal__itpcl_or_cuddalore_tpp__r0009,Tamil_Nadu,PQ,,600.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
coal__itpcl_or_cuddalore_tpp__r0010,Tamil_Nadu,PQ,,600.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
coal__mettur_tps__r0011,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
coal__mettur_tps__r0012,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
coal__mettur_tps__r0013,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
small_hydro__thirumurthy_mini_hydel_project__r0320,Tamil_Nadu,PQ,,0.65,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
small_hydro__vaigai_hydro_power_project__r0321,Tamil_Nadu,PQ,,3.00,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
small_hydro__vaigai_hydro_power_project__r0322,Tamil_Nadu,PQ,,3.00,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0


In [ ]:
display(n.storage_units)

Running

In [ ]:
status, termination_condition = n.optimize(
    solver_name="highs",
    formulation="kirchhoff",
)

if termination_condition != "optimal":
    raise RuntimeError(f"Optimization failed: {status}, {termination_condition}")

print(f"Optimization: {status} ({termination_condition})")

In [ ]:
n.export_to_netcdf("tamil_nadu_2025_26.nc")
